# Разведочный анализ данных 

## Предварительная настройка

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
palette = [
    "#264653", 
    "#2a9d8f", 
    "#e9c46a", 
    "#f4a261", 
    "#e76f51", 
    "#8ab17d", 
    "#b5179e"
]

## Постановка задачи

Задача состоит в прогнозировании РТО на февраль для сети магазинов пятерочка. 

Для оценки качества предсказаний будет использоваться функция следующего вида: 

$$
\text{MAPE} = 100*\frac{1}{n}*\sum_{i=1}^n
\lvert 
\frac{y_{\text{pred}_i} - y_{\text{true}_i}}{y_{\text{true}_i}}
\rvert
$$

$$
\text{score} = 100 - \text{min}(\text{MAPE}, 100)
$$

Из описания датасета:

| Название колонки | Тип данных | Описание |
| :--- | :--- | :--- |
| **store_id** | `int64` | Уникальный идентификатор магазина |
| **year** / **month** | `int64` | Год и месяц наблюдения |
| **rto** | `float64` | **Целевая переменная:** Розничный товарооборот за месяц |
| **avg_items** / **avg_promo_items**| `float64` | Среднее кол-во товаров / промо-товаров в одном чеке |
| **avg_cancels** | `float64` | Среднее количество отмен чеков |
| **working_hours** | `float64` | Количество рабочих часов магазина в день |
| **opening_date_cat** | `object` | Категория даты открытия магазина |
| **store_area_cat** | `object` | Категория торговой площади |
| **city** / **region** | `object` | Населенный пункт и регион РФ |
| **population** / **households** | `int64` | Численность населения и кол-во домохозяйств вокруг |
| **foot_traffic** / **car_traffic** | `int64` | Пешеходный и автомобильный трафик в час |
| **marketplaces_100m** | `int64` | Кол-во маркетплейсов, доставок и постаматов в радиусе 100м |
| **pharmacy_300m** / **schools_300m** | `int64` | Кол-во аптек/мед.учреждений и школ в радиусе 300м |
| **stops_300m** | `int64` | Количество остановок общественного транспорта в радиусе 300м |
| **grocery_500m** / **pyaterochka_500m**| `int64` | Кол-во продуктовых конкурентов и других Пятёрочек в радиусе 500м |
| **cash_registers** | `int64` | Количество касс в магазине |
| **alcohol_license** | `int64` | Флаг наличия алкогольной лицензии (0 или 1) |

In [ ]:
def get_score(y_true: np.ndarray, predictions: np.ndarray) -> tuple:
    eps_cons = 1e-8
    mape = 100 * np.mean(
        np.abs((predictions - y_true) / np.maximum(y_true, eps_cons))
    )
    score = 100 * ((100 - min(mape, 100)) / 100) ** 2
    return mape, score

In [ ]:
rename_dict = {
    "new_id": "store_id",
    "Год": "year",
    "Месяц": "month",
    "Среднее количество промо товаров в чеке": "avg_promo_items",
    "Среднее количество товаров в чеке": "avg_items",
    "Среднее количество отмен": "avg_cancels",
    "Рабочие часы в день": "working_hours",
    "Дата открытия, категориальный": "opening_date_cat",
    "Торговая площадь, категориальный": "store_area_cat",
    "Населенный пункт": "city",
    "Регион": "region",
    "Численность населения": "population",
    "Количество домохозяйств": "households",
    "Трафик пеший, в час": "foot_traffic",
    "Трафик авто, в час": "car_traffic",
    "Маркетплейсы, доставки, постаматы (100 м)": "marketplaces_100m",
    "Медицинские уч. и аптеки (300 м)": "pharmacy_300m",
    "Школы (300 м)": "schools_300m",
    "Остановки (300 м)": "stops_300m",
    "Продуктовые магазины (500 м)": "grocery_500m",
    "Пятерочки (500 м)": "pyaterochka_500m",
    "Количество касс": "cash_registers",
    "Флаг алкогольной лицензии": "alcohol_license",
    "РТО": "rto"
}

## EDA

In [ ]:
df = pd.read_csv("train_2.csv")
df = df.rename(columns=rename_dict)
df.sample(5)

In [ ]:
print(f"Размер датасета: {df.shape[0]}")
print(f"Количество колонок: {df.shape[1]}")
print(f"Наличие дубликатов: {df.duplicated().any()}\n")
print(f"Наличие пропусков:\n{df.isna().sum().sort_values(ascending=False)}")

In [ ]:
def get_stats(X):
    uniques = X.nunique()
    return pd.DataFrame({
        "Количество уникальных": uniques,
        "Доля": np.round(uniques / X.shape[0] * 100, 2),
        "Тип данных": X.dtypes
    }).sort_values(by="Доля", ascending=False)

In [ ]:
get_stats(df).head(10)